In [3]:
%pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 torchtext==0.17.0 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 GB 4.2 MB/s eta 0:09:46
     ---------------------------------------- 0.0/2.5 GB 4.9 MB/s eta 0:08:16
     ---------------------------------------- 0.0/2.5 GB 4.6 MB/s eta 0:08:56
     ---------------------------------------- 0.0/2.5 GB 4.4 MB/s eta 0:09:18
     ---------------------------------------- 0.0/2.5 GB 4.1 MB/s eta 0:10:04
     ---------------------------------------- 0.0/2.5 GB 4.0 MB/s eta 0:10:17
     ---------------------------------------- 0.0/2.5 GB 4.0 MB/s eta 0:10:19
     ---------------------------------------- 0.0/2.5 GB 3.9 MB/s eta 0:10:20
     ---------------------------------------- 0.0/2.5 GB 3.9 MB/s eta 0:10:22
     ---------------------------------------- 0.0/2.5 GB 3.8 MB/s eta 0:10:49
     ---------------------------------------- 0.0/2.5 GB 3.9 MB/s eta 0:10:23
     -------

In [6]:
%pip install spacy

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached exceptiongroup-1.3.1-py3-none-any.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.4 MB 4.6 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.4 MB 4.6 MB/s eta 0:00:04
   --- ------------------------------------ 1.3/16.4 MB 1.8 MB/s eta 0:00:09
   ---- ---------------------------

In [15]:
%pip install nltk

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 4.1 MB/s  0:00:00

   -------------------- ------------------- 2/4 [joblib]
   -------------------- ------------------- 2/4 [joblib]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ----

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import nltk

c:\Users\shand\anaconda3\envs\seq2seq_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)


In [3]:
dataset=datasets.load_dataset('bentrevett/multi30k')
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [4]:
train_df,val_df,test_df=dataset["train"],dataset["validation"],dataset["test"]
print(train_df[0],val_df[0],test_df[0])

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'} {'en': 'A group of men are loading cotton onto a truck', 'de': 'Eine Gruppe von Männern lädt Baumwolle auf einen Lastwagen'} {'en': 'A man in an orange hat starring at something.', 'de': 'Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.'}


### Tokenize

###### Will be using spacy models which has the tokenizers, using these tokenizers we will try to transform the sentences in dataset into tokens. 

In [5]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

In [6]:
## WE can use the .tokenizer function on any string that returns sequence of token objects

print(en_nlp.tokenizer("WE can use the .tokenizer function on any string that returns sequence of token objects"))
## WE can use the .tokenizer function on any string that returns sequence of token objects

print(en_nlp.tokenizer("WE can use the .tokenizer function on any string that returns sequence of token objects")[0])
## WE can use the .tokenizer function on any string that returns sequence of token objects

en_nlp.tokenizer("WE can use the .tokenizer function on any string that returns sequence of token objects")[0].text



WE can use the .tokenizer function on any string that returns sequence of token objects
WE


'WE'

In [7]:
en_nlp.tokenizer("WE can use the .tokenizer function on any string that returns sequence of token objects")[0]

WE

In [8]:
def tokenize_sentences(sentence,en_nlp,eos,sos,de_nlp,maximum_length,lower):
    en_tokens=[token.text for token in en_nlp.tokenizer(sentence['en'])]
    de_tokens=[token.text for token in de_nlp.tokenizer(sentence['de'])]
    en_tokens=[sos]+en_tokens+[eos]
    de_tokens=[sos]+de_tokens+[eos]
    
    return {"en_tokens":en_tokens,"de_tokens":de_tokens}
    

In [9]:
fn_args={
    "en_nlp":en_nlp,
    "de_nlp":de_nlp,
    "eos":"<eos>",
    "sos":"<sos>",
    "maximum_length":1000,
    "lower":True,
}

train_data=train_df.map(tokenize_sentences,fn_kwargs=fn_args)
val_data=val_df.map(tokenize_sentences,fn_kwargs=fn_args)
test_data=test_df.map(tokenize_sentences,fn_kwargs=fn_args)


Map: 100%|██████████| 1000/1000 [00:00<00:00, 4474.56 examples/s]


In [10]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'Two',
  'young',
  ',',
  'White',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'Zwei',
  'junge',
  'weiße',
  'Männer',
  'sind',
  'im',
  'Freien',
  'in',
  'der',
  'Nähe',
  'vieler',
  'Büsche',
  '.',
  '<eos>']}

##### Building Vocubalary

In [11]:
minimum_freq=2
unk_token="<unk>"
pad_token="<pad>"
eos_token="<eos>"
sos_token="<sos>"

special_tokens=[eos_token,sos_token,unk_token,pad_token]

en_vocab=torchtext.vocab.build_vocab_from_iterator(train_data["en_tokens"],min_freq=minimum_freq,specials=special_tokens)
de_vocab=torchtext.vocab.build_vocab_from_iterator(train_data["de_tokens"],min_freq=minimum_freq,specials=special_tokens)


In [12]:
en_vocab.get_itos()#Returns list of words 
en_vocab['A'] #returns the index of specified token
en_vocab.get_stoi()["the"]

8

In [13]:
#We havent fixed the problem when we try to access a token that is not present in the vocabulary: The fix is that we can return the index of unknown token
assert en_vocab[unk_token]==de_vocab[unk_token]
assert en_vocab[pad_token]==de_vocab[pad_token]

unk_index=en_vocab[unk_token]
pad_index=en_vocab[pad_token]


In [14]:
unk_index

2

In [15]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)


In [16]:
en_vocab["dasdasdwadw"]

2

In [17]:
#lookup_indices() method. This takes in a list of tokens and returns a list of indices.

tokens=["i","love","playing","cricket"]
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))


['i', 'love', 'playing', 'cricket']

In [18]:
def numericalize_sentence(sentence,en_vocab,de_vocab):
    en_indices=en_vocab.lookup_indices(sentence["en_tokens"])
    de_indices=de_vocab.lookup_indices(sentence["de_tokens"])
    return {"en_indices":en_indices,"de_indices":de_indices}
args={"en_vocab":en_vocab,"de_vocab":de_vocab}

train_data=train_data.map(numericalize_sentence,fn_kwargs=args)
val_data=val_data.map(numericalize_sentence,fn_kwargs=args)
test_data=test_data.map(numericalize_sentence,fn_kwargs=args)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 11644.31 examples/s]


In [19]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'Two',
  'young',
  ',',
  'White',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'Zwei',
  'junge',
  'weiße',
  'Männer',
  'sind',
  'im',
  'Freien',
  'in',
  'der',
  'Nähe',
  'vieler',
  'Büsche',
  '.',
  '<eos>'],
 'en_indices': [1, 19, 25, 15, 1169, 808, 17, 57, 84, 336, 1339, 5, 0],
 'de_indices': [1, 21, 85, 257, 31, 87, 22, 94, 7, 16, 112, 7910, 3209, 4, 0]}

In [20]:
datatype="torch"
req_columns=["en_indices","de_indices"]

train_data=train_data.with_format(
    type=datatype,columns=req_columns,output_all_columns=True
)
val_data=val_data.with_format(
    type=datatype,columns=req_columns,output_all_columns=True
)

test_data=test_data.with_format(
    type=datatype,columns=req_columns,output_all_columns=True
)


In [21]:
train_data[0]

{'en_indices': tensor([   1,   19,   25,   15, 1169,  808,   17,   57,   84,  336, 1339,    5,
            0]),
 'de_indices': tensor([   1,   21,   85,  257,   31,   87,   22,   94,    7,   16,  112, 7910,
         3209,    4,    0]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'Two',
  'young',
  ',',
  'White',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'Zwei',
  'junge',
  'weiße',
  'Männer',
  'sind',
  'im',
  'Freien',
  'in',
  'der',
  'Nähe',
  'vieler',
  'Büsche',
  '.',
  '<eos>']}

In [44]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_indices=[sentence["en_indices"] for sentence in batch]
        batch_de_indices=[sentence["de_indices"] for sentence in batch]
        
        batch_en_indices=nn.utils.rnn.pad_sequence(batch_en_indices,padding_value=pad_index)
        batch_de_indices=nn.utils.rnn.pad_sequence(batch_de_indices,padding_value=pad_index)
        batch={
            "en_indices":batch_en_indices,
            "de_indices":batch_de_indices,
        }
        return batch
    return collate_fn
        

In [45]:
def get_data_loader(dataset,batch_size,pad_index,shuffle=False):
    collate_fn=get_collate_fn(pad_index)
    data_loader=torch.utils.data.DataLoader(
        dataset,batch_size=batch_size,collate_fn=collate_fn,shuffle=shuffle
    )
    return data_loader


In [46]:
batch_size=32

train_data_loader=get_data_loader(train_data,batch_size,pad_index,shuffle=True)
val_data_loader=get_data_loader(val_data,batch_size,pad_index,shuffle=True)
test_data_loader=get_data_loader(test_data,batch_size,pad_index,shuffle=True)

In [57]:

class Encoder(nn.Module):
    def __init__(self,input_dim,hidden_dim,embedding_dim,dropout,num_layers=1):
        super().__init__()
        self.hidden_dim=hidden_dim
        self.input_dim=input_dim
        self.n_layers=num_layers
        self.embeddings=nn.Embedding(input_dim,embedding_dim)
        self.rnn=nn.LSTM(embedding_dim,hidden_dim,num_layers,dropout=dropout)
        self.dropout=nn.Dropout(dropout)
    
    def forward(self,src):
        embedded=self.embeddings(src)
        outputs,(hidden,cell)=self.rnn(self.dropout(embedded))
        return hidden,cell

class Decoder(nn.Module):
    def __init__(self,output_dim,hidden_dim,embedding_dim,dropout,num_layers=1):
        super().__init__()
        self.output_dim=output_dim
        self.hidden_dim=hidden_dim
        self.n_layers=num_layers
        self.embeddings=nn.Embedding(output_dim,embedding_dim)
        self.rnn=nn.LSTM(embedding_dim,hidden_dim,num_layers,dropout=dropout)
        self.fc_out=nn.Linear(hidden_dim,output_dim)
        self.dropout=nn.Dropout(dropout)               
    def forward(self,inp,hidden,cell):
        inp=inp.unsqueeze(0)
        embedded=self.dropout(self.embeddings(inp))
        output,(hidden,cell)=self.rnn(embedded,(hidden,cell))
        output=output.squeeze(0)
        prediction=self.fc_out(output)  
        return prediction,hidden,cell
        
    
    






In [58]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = trg[0, :]   
        for t in range(1, trg_length):
            output, hidden, cell = self.decoder(input, hidden, cell)

            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[t] if teacher_force else top1

        return outputs

In [59]:
tensor_V=torch.zeros(10,128,10000)
tensor=tensor_V[0]
tensor.shape

torch.Size([128, 10000])

In [60]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    hidden_dim,
    encoder_embedding_dim,
    encoder_dropout,
    n_layers,
)

decoder = Decoder(
    output_dim,
    hidden_dim,
    decoder_embedding_dim,
    decoder_dropout,
    n_layers,
)

model = Seq2Seq(encoder, decoder, device).to(device)

In [61]:
total_weights=0

for name,parameters in model.named_parameters():
    print(name)
    total_weights += parameters.numel()

encoder.embeddings.weight
encoder.rnn.weight_ih_l0
encoder.rnn.weight_hh_l0
encoder.rnn.bias_ih_l0
encoder.rnn.bias_hh_l0
encoder.rnn.weight_ih_l1
encoder.rnn.weight_hh_l1
encoder.rnn.bias_ih_l1
encoder.rnn.bias_hh_l1
decoder.embeddings.weight
decoder.rnn.weight_ih_l0
decoder.rnn.weight_hh_l0
decoder.rnn.bias_ih_l0
decoder.rnn.bias_hh_l0
decoder.rnn.weight_ih_l1
decoder.rnn.weight_hh_l1
decoder.rnn.bias_ih_l1
decoder.rnn.bias_hh_l1
decoder.fc_out.weight
decoder.fc_out.bias


In [62]:
def initialize_weights(m):
    for name,parameters in m.named_parameters():
        nn.init.uniform_(parameters.data,-0.08,0.08)
        
model.apply(initialize_weights)
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

In [63]:
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

In [64]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_indices"].to(device)
        trg = batch["en_indices"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [65]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_indices"].to(device)
            trg = batch["en_indices"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)


In [ ]:
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        val_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

 10%|█         | 1/10 [02:36<23:24, 156.02s/it]

	Train Loss:   3.990 | Train PPL:  54.031
	Valid Loss:   4.326 | Valid PPL:  75.616


 20%|██        | 2/10 [04:02<15:22, 115.35s/it]

	Train Loss:   3.701 | Train PPL:  40.489
	Valid Loss:   4.170 | Valid PPL:  64.726


 30%|███       | 3/10 [06:51<16:16, 139.48s/it]

...

	Train Loss:   3.269 | Train PPL:  26.280
	Valid Loss:   3.923 | Valid PPL:  50.547


 50%|█████     | 5/10 [11:23<10:58, 131.72s/it]

	Train Loss:   3.083 | Train PPL:  21.834
	Valid Loss:   3.858 | Valid PPL:  47.393


 60%|██████    | 6/10 [12:20<07:04, 106.15s/it]

	Train Loss:   2.930 | Train PPL:  18.728
	Valid Loss:   3.785 | Valid PPL:  44.053


In [ ]:
model.load_state_dict(torch.load("tut1-model.pt"))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")